# UD03 Tarea con gemini

## Enunciado

Ahora que ya conocemos todo el ecosistema Hadoop-Spark en profundidad, vamos a hacer un ejercicio que abarque todo el proceso de tratamiento de los datos: desde que se extraen de un repositorio, se limpian, se trasforman, se exportan a una BD (en este caso relacional) y se realizan consultas sobre ellos.

Para ello una vez descargado el data set:


Debes entregar un cuaderno de Colab con:

Explicación del conjunto de datos elegido cuál es su temática, su tamaño, con qué campos se relaciona y por qué puede resultar interesante.  
Una vez quede claro el contexto:

1. Importa de manera automática tus conjuntos de datos. Realizado con archivo en drive.  
2. Usa Pandas para realizar una primera visualización de ellos en forma de tabla, y estudiar los datos: observar si hay valores nulos, fechas con formatos que no se corresponden, campos vacíos...   
3. Usa Apache Pig para  
   1. corregir los fallos en los datos que hayas observado en el punto anterior. (Al menos debes modificar dos campos, por ejemplo cambiar formatos de fechas y rellenar campos vacíos y nulos con valores por defecto)  
   2. Realizar un tratamiento de datos que consideres interesante: por ejemplo, encontrar las 3 palabras más repetidas en ambos archivos.  
4. Usa Spark (PySpark) para importar tus ficheros a una base de datos relacional Hive.  
5. Realiza al menos dos consultas HQL que impliquen dos tablas.

# Instalar Hadoop y Pig

Lo primero que hago es activar el soporte de GPU.

**"Entorno de ejecución" > "Cambiar tipo de entorno de ejecución"**

## Verificaciones previas

Primero, verifico la versión de Java:

In [1]:
!ls -l /usr/lib/jvm/

total 4
lrwxrwxrwx 1 root root   21 Jan 29 03:35 java-1.17.0-openjdk-amd64 -> java-17-openjdk-amd64
drwxr-xr-x 9 root root 4096 Apr  2 13:13 java-17-openjdk-amd64


## Instalación de Hadoop

Entro a https://downloads.apache.org/hadoop/common y veo las versiones disponibles para descargar.

Una vez comprobado qué versión de Java necesito para la versión de Hadoop que usaré, descargo el tar.gz correspondiente.

Para saberlo, consulto el archivo Changelog y busco "Java". Ahí veo frases como:

"[JDK17] Add JPMS options required by Java 17+"

In [2]:
import os

# Download and install Hadoop 3.4.2
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz
!tar -xzf hadoop-3.4.2.tar.gz
# Remove destination if it exists to avoid errors or nesting
!rm -rf /usr/local/hadoop-3.4.2
!mv hadoop-3.4.2 /usr/local/
!rm hadoop-3.4.2.tar.gz

# Update environment variables for Java 17 and Hadoop 3.4.2
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop-3.4.2"

# Update PATH to include Hadoop bin
if "/usr/local/hadoop-3.4.2/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = os.environ["PATH"] + ":" + "/usr/local/hadoop-3.4.2/bin"

# Verify installations
print("Hadoop Version:")
!hadoop version
print("\nJava Version:")
!java -version

Hadoop Version:
Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /usr/local/hadoop-3.4.2/share/hadoop/common/hadoop-common-3.4.2.jar

Java Version:
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [3]:
ls -l $HADOOP_HOME/etc/hadoop

total 184
-rw-r--r-- 1 1001 1001  9213 Aug 20  2025 capacity-scheduler.xml
-rw-r--r-- 1 1001 1001  1335 Aug 20  2025 configuration.xsl
-rw-r--r-- 1 1001 1001  2567 Aug 20  2025 container-executor.cfg
-rw-r--r-- 1 1001 1001   774 Aug 20  2025 core-site.xml
-rw-r--r-- 1 1001 1001  3999 Aug 20  2025 hadoop-env.cmd
-rw-r--r-- 1 1001 1001 16786 Aug 20  2025 hadoop-env.sh
-rw-r--r-- 1 1001 1001  3321 Aug 20  2025 hadoop-metrics2.properties
-rw-r--r-- 1 1001 1001 14007 Aug 20  2025 hadoop-policy.xml
-rw-r--r-- 1 1001 1001  3414 Aug 20  2025 hadoop-user-functions.sh.example
-rw-r--r-- 1 1001 1001   683 Aug 20  2025 hdfs-rbf-site.xml
-rw-r--r-- 1 1001 1001   775 Aug 20  2025 hdfs-site.xml
-rw-r--r-- 1 1001 1001  1484 Aug 20  2025 httpfs-env.sh
-rw-r--r-- 1 1001 1001  1657 Aug 20  2025 httpfs-log4j.properties
-rw-r--r-- 1 1001 1001   620 Aug 20  2025 httpfs-site.xml
-rw-r--r-- 1 1001 1001  3518 Aug 20  2025 kms-acls.xml
-rw-r--r-- 1 1001 1001  1351 Aug 20  2025 kms-env.sh
-rw-r--r-- 1 1001 1001 

## Instalación de Pig y configuración de entrono

In [4]:
%%bash
wget https://downloads.apache.org/pig/pig-0.17.0/pig-0.17.0.tar.gz
tar -xzf pig-0.17.0.tar.gz
cp -r pig-0.17.0/ /usr/local/

--2026-04-22 17:52:52--  https://downloads.apache.org/pig/pig-0.17.0/pig-0.17.0.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f8:10a:39da::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 230606579 (220M) [application/x-gzip]
Saving to: ‘pig-0.17.0.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  219K 17m6s
    50K .......... .......... .......... .......... ..........  0% 1.08M 10m15s
   100K .......... .......... .......... .......... ..........  0%  646K 8m46s
   150K .......... .......... .......... .......... ..........  0% 13.0M 6m39s
   200K .......... .......... .......... .......... ..........  0%  521K 6m45s
   250K .......... .......... .......... .......... ..........  0% 16.5M 5m40s
   300K .......... .......... .......... .......... ..........  0% 4.82M 4m58s
   350K .......... .........

Establecemos las variables de entorno para Pig

In [5]:
import os
# Update PIG_CLASSPATH
os.environ["PIG_HOME"] = "/usr/local/pig-0.17.0"
os.environ["PATH"] = os.environ["PATH"] + ":" + "/usr/local/pig-0.17.0/bin"
os.environ["PIG_CLASSPATH"] = "/usr/local/hadoop-3.4.2/etc/hadoop"

Verificamos la instalación

In [6]:
!pig -h -version


Apache Pig version 0.17.0 (r1797386) 
compiled Jun 02 2017, 15:41:58

USAGE: Pig [options] [-] : Run interactively in grunt shell.
       Pig [options] -e[xecute] cmd [cmd ...] : Run cmd(s).
       Pig [options] [-f[ile]] file : Run cmds found in file.
  options include:
    -4, -log4jconf - Log4j configuration file, overrides log conf
    -b, -brief - Brief logging (no timestamps)
    -c, -check - Syntax check
    -d, -debug - Debug level, INFO is default
    -e, -execute - Commands to execute (within quotes)
    -f, -file - Path to the script to execute
    -g, -embedded - ScriptEngine classname or keyword for the ScriptEngine
    -h, -help - Display this message. You can specify topic to get help for that topic.
        properties is the only topic currently supported: -h properties.
    -i, -version - Display version information
    -l, -logfile - Path to client side log file; default is current working directory.
    -m, -param_file - Path to the parameter file
    -p, -param - K

# 1.- **Apartado de Importación de datos**
---
Importa de manera automática tus conjuntos de datos desde Kaggle u otro repositorio (Pista: puedes usar las librerías kaggle, opendatests u otras opciones que consideres más oportunas, también puedes usar squoop, dependiendo de dónde se encuentren tus datos).
---
---

## Clonar repo con dataset (alternative para examen)

In [7]:
!git clone https://github.com/kachytronico/BDA_examen_26

Cloning into 'BDA_examen_26'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 74 (delta 22), reused 62 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 938.38 KiB | 7.82 MiB/s, done.
Resolving deltas: 100% (22/22), done.


## Descarga desde archivo de Drive

Con el bloque de código que aparece a continuación, descargamos los ficheros que hemos alojado en nuestro Google Drive y los ubicamos en la ruta de Colab que vamos a emplear en los próximos pasos.

Además de los tres archivos CSV mencionados en el punto anterior, como el fichero *transactions_data.csv* tiene **más de 13 millones de líneas** y puede dar problemas de memoria RAM o tardar mucho en la ejecución, hemos subido también una versión CORTA.

In [8]:
# Descargamos un fichero comprimido con cuatro archivos desde Google Drive usando el ID del archivo
!gdown --id 1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9 -O datasets.zip

# Descomprimimos el archivo ZIP descargado en la carpeta "datasets"
!unzip datasets.zip -d datasets/

import os

# Eliminamos el archivo ZIP descargado
if os.path.exists("datasets.zip"):
    os.remove("datasets.zip")
    print("El archivo datasets.zip ha sido eliminado.")
else:
    print("El archivo datasets.zip no existe.")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9
From (redirected): https://drive.google.com/uc?id=1_6ptGvYG_tgllD3axsE6FQT4ZYWxgKJ9&confirm=t&uuid=db53a354-ed93-49d9-82fd-16b2d8d47e20
To: /content/datasets.zip
100% 312M/312M [00:03<00:00, 85.6MB/s]
Archive:  datasets.zip
  inflating: datasets/users_data.csv  
  inflating: datasets/cards_data.csv  
  inflating: datasets/transactions_data.csv  
  inflating: datasets/transactionsCORTO_data.csv  
El archivo datasets.zip ha sido eliminado.


### Comprobación de archivos importados

Verificamos que los archivos CSV se encuentran en la carpeta `datasets`.

In [9]:
import os

# Listar el contenido del directorio 'datasets'
files_in_datasets = os.listdir('datasets')
print("Archivos en la carpeta 'datasets':")
for file_name in files_in_datasets:
    print(file_name)

Archivos en la carpeta 'datasets':
cards_data.csv
transactions_data.csv
users_data.csv
transactionsCORTO_data.csv


## Mover Datasets Locales a HDFS

Ahora que Hadoop y Pig están instalados y configurados, el siguiente paso es mover los archivos CSV que hemos descargado localmente a un directorio en HDFS, en este caso, llamado `/input`. Esto es necesario para que Pig pueda procesarlos.

In [10]:
# Crear el directorio /input en HDFS
!hdfs dfs -mkdir -p /input

# Copiar los archivos desde la ruta local /content/datasets/ a /input en HDFS
!hdfs dfs -put /content/datasets/* /input

# Verificar que los archivos se encuentran correctamente en HDFS
!hdfs dfs -ls /input

Found 4 items
-rw-r--r--   1 root root     509860 2026-04-22 17:53 /input/cards_data.csv
-rw-r--r--   1 root root     951089 2026-04-22 17:53 /input/transactionsCORTO_data.csv
-rw-r--r--   1 root root 1258531040 2026-04-22 17:53 /input/transactions_data.csv
-rw-r--r--   1 root root     164831 2026-04-22 17:53 /input/users_data.csv


### Visualización de las primeras líneas de los archivos en HDFS

In [11]:
# Mostrar las primeras líneas de cards_data.csv
!hdfs dfs -cat /input/cards_data.csv | head -n 5

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
cat: Unable to write to output stream.


In [12]:
# Mostrar las primeras líneas de transactionsCORTO_data.csv
!hdfs dfs -cat /input/transactionsCORTO_data.csv | head -n 5

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
cat: Unable to write to output stream.


In [13]:
# Mostrar las primeras líneas de transactions_data.csv
!hdfs dfs -cat /input/transactions_data.csv | head -n 5

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
cat: Unable to write to output stream.


In [14]:
# Mostrar las primeras líneas de users_data.csv
!hdfs dfs -cat /input/users_data.csv | head -n 5

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
cat: Unable to write to output stream.


### Conclusión del Punto 1: Importación de Datos

El primer punto del enunciado, referente a la importación automática de los conjuntos de datos, se ha completado satisfactoriamente.

Se han realizado los siguientes pasos:

1.  **Descarga y Extracción:** Los archivos de datos (`users_data.csv`, `cards_data.csv`, `transactions_data.csv`, y `transactionsCORTO_data.csv`) han sido descargados de Google Drive y descomprimidos en el entorno local de Colab dentro de la carpeta `/content/datasets/`.
2.  **Verificación Local:** Se ha confirmado la existencia de los archivos en la ruta local.
3.  **Transferencia a HDFS:** Todos los archivos CSV han sido copiados exitosamente desde la ubicación local (`/content/datasets/`) al sistema de archivos distribuido de Hadoop (HDFS) en el directorio `/input`.
4.  **Verificación en HDFS:** Se ha verificado que los archivos están correctamente listados en HDFS y se han mostrado las primeras líneas de cada uno para una inspección inicial del formato y contenido.

Con esto, los datos están ahora disponibles en HDFS para ser procesados por herramientas como Pig y Spark, cumpliendo con el requisito de importación del ejercicio.

# 2.- Primera Visualización y Estudio de Datos con Pandas

En este apartado, utilizaremos la librería Pandas para cargar los datasets en DataFrames, visualizarlos y realizar una primera inspección para identificar posibles problemas como valores nulos, formatos incorrectos o campos vacíos.

In [15]:
import pandas as pd

# Función para leer archivos de HDFS a un DataFrame local
def read_hdfs_csv_to_pandas(hdfs_path, local_path, separator=','):
    !hdfs dfs -get -f {hdfs_path} {local_path}
    df = pd.read_csv(local_path, sep=separator)
    # Limpiar el archivo local después de leerlo para ahorrar espacio
    !rm {local_path}
    return df

### Inspección de `cards_data.csv`

In [16]:
print('Cargando cards_data.csv...')
cards_df = read_hdfs_csv_to_pandas('/input/cards_data.csv', 'cards_data.csv')

print('\nPrimeras 5 filas de cards_data.csv:')
print(cards_df.head())

print('\nInformación del DataFrame cards_data.csv:')
cards_df.info()

print('\nValores nulos en cards_data.csv:')
print(cards_df.isnull().sum())

Cargando cards_data.csv...

Primeras 5 filas de cards_data.csv:
     id  client_id  card_brand        card_type       card_number  expires  \
0  4524        825        Visa            Debit  4344676511950444  12/2022   
1  2731        825        Visa            Debit  4956965974959986  12/2020   
2  3701        825        Visa            Debit  4582313478255491  02/2024   
3    42        825        Visa           Credit  4879494103069057  08/2024   
4  4659        825  Mastercard  Debit (Prepaid)  5722874738736011  03/2009   

   cvv has_chip  num_cards_issued credit_limit acct_open_date  \
0  623      YES                 2       $24295        09/2002   
1  393      YES                 2       $21968        04/2014   
2  719      YES                 2       $46414        07/2003   
3  693       NO                 1       $12400        01/2003   
4   75      YES                 1          $28        09/2008   

   year_pin_last_changed card_on_dark_web  
0                   2008        

### Inspección de `users_data.csv`

In [17]:
print('Cargando users_data.csv...')
users_df = read_hdfs_csv_to_pandas('/input/users_data.csv', 'users_data.csv')

print('\nPrimeras 5 filas de users_data.csv:')
print(users_df.head())

print('\nInformación del DataFrame users_data.csv:')
users_df.info()

print('\nValores nulos en users_data.csv:')
print(users_df.isnull().sum())

Cargando users_data.csv...

Primeras 5 filas de users_data.csv:
     id  current_age  retirement_age  birth_year  birth_month  gender  \
0   825           53              66        1966           11  Female   
1  1746           53              68        1966           12  Female   
2  1718           81              67        1938           11  Female   
3   708           63              63        1957            1  Female   
4  1164           43              70        1976            9    Male   

                    address  latitude  longitude per_capita_income  \
0             462 Rose Lane     34.15    -117.76            $29278   
1    3606 Federal Boulevard     40.76     -73.74            $37891   
2           766 Third Drive     34.02    -117.89            $22681   
3          3 Madison Street     40.71     -73.99           $163145   
4  9620 Valley Stream Drive     37.76    -122.44            $53797   

  yearly_income total_debt  credit_score  num_credit_cards  
0        $59696

### Inspección de `transactionsCORTO_data.csv`

Utilizaremos la versión 'CORTO' del dataset de transacciones para la inspección inicial con Pandas, debido al gran tamaño del archivo `transactions_data.csv` completo, que podría causar problemas de memoria en este entorno.

In [18]:
print('Cargando transactionsCORTO_data.csv...')
transactions_corto_df = read_hdfs_csv_to_pandas('/input/transactionsCORTO_data.csv', 'transactionsCORTO_data.csv')

print('\nPrimeras 5 filas de transactionsCORTO_data.csv:')
print(transactions_corto_df.head())

print('\nInformación del DataFrame transactionsCORTO_data.csv:')
transactions_corto_df.info()

print('\nValores nulos en transactionsCORTO_data.csv:')
print(transactions_corto_df.isnull().sum())

Cargando transactionsCORTO_data.csv...

Primeras 5 filas de transactionsCORTO_data.csv:
        id                 date  client_id  card_id   amount  \
0  7475327  2010-01-01 00:01:00       1556     2972  $-77.00   
1  7475328  2010-01-01 00:02:00        561     4575   $14.57   
2  7475329  2010-01-01 00:02:00       1129      102   $80.00   
3  7475331  2010-01-01 00:05:00        430     2860  $200.00   
4  7475332  2010-01-01 00:06:00        848     3915   $46.41   

            use_chip  merchant_id merchant_city merchant_state      zip   mcc  \
0  Swipe Transaction        59935        Beulah             ND  58523.0  5499   
1  Swipe Transaction        67570    Bettendorf             IA  52722.0  5311   
2  Swipe Transaction        27092         Vista             CA  92084.0  4829   
3  Swipe Transaction        27092   Crown Point             IN  46307.0  4829   
4  Swipe Transaction        13051       Harwood             MD  20776.0  5813   

  errors  
0    NaN  
1    NaN  
2    Na

### Conclusión del Punto 2: Primera Visualización y Estudio de Datos

Tras la primera visualización y estudio de los datasets con Pandas, se han identificado los siguientes puntos clave:

*   **`cards_data.csv` y `users_data.csv`:** Ambos datasets no presentan valores nulos. Sin embargo, contienen columnas como `expires`, `acct_open_date`, `credit_limit`, `per_capita_income`, `yearly_income` y `total_debt` que son de tipo `object` (cadena de texto) y que requieren ser convertidas a tipos de datos más adecuados (fechas o numéricos) para su correcto análisis. Los campos monetarios (`credit_limit`, `per_capita_income`, `yearly_income`, `total_debt`) necesitan que se elimine el símbolo '$' antes de la conversión.

*   **`transactionsCORTO_data.csv`:** Este dataset muestra varios problemas:
    *   **Valores Nulos:** Las columnas `merchant_state` (1179 nulos), `zip` (1205 nulos) y `errors` (9848 nulos) contienen una cantidad significativa de valores faltantes que deberán ser tratados (rellenados o eliminados).
    *   **Tipos de Datos y Formato:** Las columnas `date` (tipo `object`) y `amount` (tipo `object` debido al símbolo '$' y signos negativos) necesitan ser convertidas a tipos de datos apropiados (fecha/hora y numérico, respectivamente).

En resumen, el siguiente paso será corregir estos problemas de calidad de datos, especialmente la gestión de valores nulos y la conversión de tipos, antes de proceder con análisis más profundos.

# 3.- Uso de Apache Pig para Corrección y Tratamiento de Datos

En este apartado, utilizaremos Apache Pig para corregir algunos de los fallos identificados en la inspección de datos del punto anterior y realizar un tratamiento de datos interesante. Nos centraremos principalmente en el archivo `transactionsCORTO_data.csv` por su tamaño manejable y la presencia de nulos y formatos inconsistentes.

### **Limpiar directorios de salida de Pig**

Antes de volver a ejecutar los scripts de Pig, es necesario eliminar los directorios de salida que se crearon en ejecuciones anteriores. Usaremos el comando `hdfs dfs -rm -r` para eliminar estos directorios.

In [19]:
!hdfs dfs -ls

Found 6 items
drwxr-xr-x   - root root       4096 2026-04-16 13:28 .config
drwxr-xr-x   - root root       4096 2026-04-22 17:53 BDA_examen_26
drwxr-xr-x   - root root       4096 2026-04-22 17:53 datasets
drwxr-xr-x   - root root       4096 2017-06-02 13:42 pig-0.17.0
-rw-r--r--   1 root root  230606579 2017-06-16 18:10 pig-0.17.0.tar.gz
drwxr-xr-x   - root root       4096 2026-04-16 13:28 sample_data


# comprobar directorios

In [33]:
!hdfs dfs -ls /input/

Found 4 items
-rw-r--r--   1 root root     509860 2026-04-22 17:53 /input/cards_data.csv
-rw-r--r--   1 root root     951089 2026-04-22 17:53 /input/transactionsCORTO_data.csv
-rw-r--r--   1 root root 1258531040 2026-04-22 17:53 /input/transactions_data.csv
-rw-r--r--   1 root root     164831 2026-04-22 17:53 /input/users_data.csv


In [21]:
!hdfs dfs -ls /output/

ls: `/output/': No such file or directory


In [22]:
!hdfs dfs -rm -r /output/transactions_cleaned

rm: `/output/transactions_cleaned': No such file or directory


Una vez ejecutada esta celda, podrás volver a ejecutar tus scripts de Pig sin que te aparezcan los errores de directorios ya existentes.

## 3.1 Corrección de Fallos en los Datos

Para `transactionsCORTO_data.csv`, realizaremos las siguientes correcciones:
*   **Campo `amount`**: Eliminar el signo '$' y convertir a tipo numérico (`float`).
*   **Campo `merchant_state`**: Rellenar valores nulos con la cadena 'UNKNOWN'.
*   **Campo `zip`**: Rellenar valores nulos con el valor numérico 0.
*   **Campo `errors`**: Rellenar valores nulos con la cadena 'NO_ERROR'.

In [23]:
%%writefile process_transactions.pig

-- Cargar el dataset de transacciones corto
transactions_raw = LOAD '/input/transactionsCORTO_data.csv'
USING PigStorage(',') AS (
    id_str:chararray,
    date:chararray,
    client_id:int,
    card_id:int,
    amount_str:chararray,
    use_chip:chararray,
    merchant_id:int,
    merchant_city:chararray,
    merchant_state_raw:chararray,
    zip_raw:chararray,
    mcc:int,
    errors_raw:chararray
);

-- Eliminar la fila de cabecera
transactions_no_header = FILTER transactions_raw BY id_str != 'id';

-- Limpiar el campo 'amount': eliminar '$', espacios y convertir a float.
-- Si amount_str es null o vacío después de limpiar '$', asignar 0.0
transactions_cleaned_amount = FOREACH transactions_no_header GENERATE
    (int)id_str AS id,
    date,
    client_id,
    card_id,
    CASE
        WHEN amount_str IS NULL THEN 0.0
        WHEN TRIM(REPLACE(amount_str, '$', '')) MATCHES '' THEN 0.0
        ELSE (float)REPLACE(TRIM(amount_str), '$', '')
    END AS amount,
    use_chip,
    merchant_id,
    merchant_city,
    merchant_state_raw,
    zip_raw,
    mcc,
    errors_raw;

-- Manejar valores nulos en merchant_state, zip y errors
transactions_filled_nulls = FOREACH transactions_cleaned_amount GENERATE
    id,
    date,
    client_id,
    card_id,
    amount,
    use_chip,
    merchant_id,
    merchant_city,
    (merchant_state_raw is null ? 'UNKNOWN' : merchant_state_raw) AS merchant_state,
    CASE
        WHEN zip_raw IS NULL THEN '0'
        WHEN TRIM(zip_raw) MATCHES '' THEN '0'
        ELSE zip_raw
    END AS zip,
    mcc,
    (errors_raw is null ? 'NO_ERROR' : errors_raw) AS errors;

-- Guardar los resultados procesados en HDFS para futuras operaciones
-- Se usa un nuevo directorio de salida para no interferir con el input original
STORE transactions_filled_nulls INTO '/output/transactions_cleaned' USING PigStorage(',');

Writing process_transactions.pig


In [24]:
# Limpiar el directorio de salida anterior para asegurar una ejecución limpia
!hdfs dfs -rm -r /output/transactions_cleaned

rm: `/output/transactions_cleaned': No such file or directory


In [25]:
# Ejecutar el script Pig actualizado
!pig process_transactions.pig

2026-04-22 17:54:22,815 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-22 17:54:22,821 INFO pig.ExecTypeProvider: Trying ExecType : MAPREDUCE
2026-04-22 17:54:22,822 INFO pig.ExecTypeProvider: Picked MAPREDUCE as the ExecType
2026-04-22 17:54:22,930 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-22 17:54:22,930 [main] INFO  org.apache.pig.Main - Logging error messages to: /content/pig_1776880462912.log
2026-04-22 17:54:23,521 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /root/.pigbootup not found
2026-04-22 17:54:23,639 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-22 17:54:23,639 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop file system at: file:///
2026-04-22 17:54:23,689 [main] INFO  org.apache.pig.PigServer - Pig Script ID for the ses

### Verificación de las correcciones en el archivo de salida

Para verificar que los cambios se aplicaron correctamente y que los valores nulos o con formatos incorrectos han sido tratados, vamos a listar el contenido del directorio de salida y a visualizar las primeras líneas del archivo procesado.

In [26]:
# Listar el contenido del directorio de salida de Pig
!hdfs dfs -ls /output/transactions_cleaned

Found 2 items
-rw-r--r--   1 root root          0 2026-04-22 17:54 /output/transactions_cleaned/_SUCCESS
-rw-r--r--   1 root root     969759 2026-04-22 17:54 /output/transactions_cleaned/part-m-00000


In [27]:
# Mostrar las primeras 10 líneas del archivo procesado para verificar las correcciones
!hdfs dfs -cat /output/transactions_cleaned/part-m-00000 | head -n 10

7475327,2010-01-01 00:01:00,1556,2972,,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NO_ERROR
7475328,2010-01-01 00:02:00,561,4575,,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NO_ERROR
7475329,2010-01-01 00:02:00,1129,102,,Swipe Transaction,27092,Vista,CA,92084.0,4829,NO_ERROR
7475331,2010-01-01 00:05:00,430,2860,,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NO_ERROR
7475332,2010-01-01 00:06:00,848,3915,,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NO_ERROR
7475333,2010-01-01 00:07:00,1807,165,,Swipe Transaction,20519,Bronx,NY,10464.0,5942,NO_ERROR
7475334,2010-01-01 00:09:00,1556,2972,,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NO_ERROR
7475335,2010-01-01 00:14:00,1684,2140,,Online Transaction,39021,ONLINE,UNKNOWN,0,4784,NO_ERROR
7475336,2010-01-01 00:21:00,335,5131,,Online Transaction,50292,ONLINE,UNKNOWN,0,7801,NO_ERROR
7475337,2010-01-01 00:21:00,351,1112,,Swipe Transaction,3864,Flushing,NY,11355.0,5813,NO_ERROR
cat: Unable to write to output stream.


## 3.2 Tratamiento de Datos con Pig: Palabras más Repetidas

En este apartado, realizaremos un tratamiento de datos interesante utilizando Apache Pig. El objetivo es encontrar las 3 palabras más repetidas en los campos textuales de los datasets `transactions_cleaned` (el resultado del procesamiento anterior) y `cards_data.csv`.

### Script Pig para encontrar las palabras más repetidas

In [28]:
%%writefile top_words_analysis.pig

-- Definir un conjunto de stopwords comunes en español e inglés
DEFINE STOPWORDS_EN (group ( 'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'he', 'in', 'is', 'it', 'its', 'of', 'on', 'that', 'the', 'to', 'was', 'were', 'will', 'with', 'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', 'couldn', 'didn', 'doesn', 'hadn', 'hasn', 'haven', 'isn', 'ma', 'mightn', 'mustn', 'needn', 'shan', 'shouldn', 'wasn', 'weren', 'won', 'wouldn' ) as (word:chararray) );

-- Cargar el dataset de cards_data.csv
cards_raw = LOAD '/input/cards_data.csv'
USING PigStorage(',') AS (
    id:int, client_id:int, card_brand:chararray, card_type:chararray,
    card_number:long, expires:chararray, cvv:int, has_chip:chararray,
    num_cards_issued:int, credit_limit:chararray, acct_open_date:chararray,
    year_pin_last_changed:int, card_on_dark_web:chararray
);

-- Eliminar la cabecera
cards_no_header = FILTER cards_raw BY id != 'id';

-- Extraer campos textuales de cards_data.csv y combinarlos
cards_text = FOREACH cards_no_header GENERATE
    CONCAT(CONCAT(CONCAT(card_brand, ' '), CONCAT(card_type, ' ')), CONCAT(has_chip, ' '), card_on_dark_web) AS full_text;

-- Tokenizar las palabras de cards_text
cards_words = FOREACH cards_text GENERATE FLATTEN(TOKENIZE(LOWER(full_text))) AS word;

-- Eliminar stopwords y palabras vacías/puntuación de cards_words
cards_filtered_words = FILTER cards_words BY NOT (word MATCHES '^[\s.,!?;:\[\](){}\"\’“”-]+$') AND NOT (word IN STOPWORDS_EN);

-- Cargar el dataset de transacciones limpio
transactions_cleaned_raw = LOAD '/output/transactions_cleaned/part-m-00000'
USING PigStorage(',') AS (
    id:int, date:chararray, client_id:int, card_id:int,
    amount:float, use_chip:chararray, merchant_id:int, merchant_city:chararray,
    merchant_state:chararray, zip:chararray, mcc:int, errors:chararray
);

-- Extraer campos textuales de transactions_cleaned y combinarlos
transactions_text = FOREACH transactions_cleaned_raw GENERATE
    CONCAT(CONCAT(CONCAT(use_chip, ' '), CONCAT(merchant_city, ' ')), CONCAT(merchant_state, ' '), errors) AS full_text;

-- Tokenizar las palabras de transactions_text
transactions_words = FOREACH transactions_text GENERATE FLATTEN(TOKENIZE(LOWER(full_text))) AS word;

-- Eliminar stopwords y palabras vacías/puntuación de transactions_words
transactions_filtered_words = FILTER transactions_words BY NOT (word MATCHES '^[\s.,!?;:\[\](){}\"\’“”-]+$') AND NOT (word IN STOPWORDS_EN);

-- Unir todas las palabras de ambos datasets
all_words = UNION cards_filtered_words, transactions_filtered_words;

-- Agrupar por palabra y contar ocurrencias
word_counts = GROUP all_words BY word;
counted_words = FOREACH word_counts GENERATE group AS word, COUNT(all_words) AS count;

-- Ordenar por conteo en orden descendente y obtener el top 3
sorted_words = ORDER counted_words BY count DESC;
top_3_words = LIMIT sorted_words 3;

-- Guardar el resultado en HDFS
STORE top_3_words INTO '/output/top_3_words' USING PigStorage(',');

Writing top_words_analysis.pig


In [29]:
# Limpiar el directorio de salida anterior para asegurar una ejecución limpia
!hdfs dfs -rm -r /output/top_3_words

rm: `/output/top_3_words': No such file or directory


In [30]:
# Ejecutar el script Pig corregido
!pig top_words_analysis.pig

2026-04-22 17:54:41,436 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-22 17:54:41,438 INFO pig.ExecTypeProvider: Trying ExecType : MAPREDUCE
2026-04-22 17:54:41,438 INFO pig.ExecTypeProvider: Picked MAPREDUCE as the ExecType
2026-04-22 17:54:41,553 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-22 17:54:41,553 [main] INFO  org.apache.pig.Main - Logging error messages to: /content/pig_1776880481532.log
2026-04-22 17:54:42,510 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /root/.pigbootup not found
2026-04-22 17:54:42,710 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-22 17:54:42,713 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop file system at: file:///
2026-04-22 17:54:42,830 [main] INFO  org.apache.pig.PigServer - Pig Script ID for the ses

In [31]:
# Crear el directorio /user/root en HDFS si no existe
!hdfs dfs -mkdir -p /user/root

# Copiar el archivo de stopwords a HDFS
!hdfs dfs -put -f stopwords_en.txt /user/root/stopwords_en.txt

put: `stopwords_en.txt': No such file or directory


Ahora modificamos el script `top_words_analysis.pig` para cargar estas stopwords y usarlas correctamente.

In [32]:
%%writefile top_words_analysis.pig

-- Cargar stopwords desde un archivo en HDFS
stopwords = LOAD '/user/root/stopwords_en.txt' USING PigStorage() AS (word:chararray);

-- Cargar el dataset de cards_data.csv
cards_raw = LOAD '/input/cards_data.csv'
USING PigStorage(',') AS (
    id:int, client_id:int, card_brand:chararray, card_type:chararray,
    card_number:long, expires:chararray, cvv:int, has_chip:chararray,
    num_cards_issued:int, credit_limit:chararray, acct_open_date:chararray,
    year_pin_last_changed:int, card_on_dark_web:chararray
);

-- Eliminar la cabecera
cards_no_header = FILTER cards_raw BY id != 'id';

-- Extraer campos textuales de cards_data.csv y combinarlos
cards_text = FOREACH cards_no_header GENERATE
    CONCAT(CONCAT(CONCAT(card_brand, ' '), CONCAT(card_type, ' ')), CONCAT(has_chip, ' '), card_on_dark_web) AS full_text;

-- Tokenizar las palabras de cards_text
cards_words = FOREACH cards_text GENERATE FLATTEN(TOKENIZE(LOWER(full_text))) AS word;

-- Eliminar stopwords y palabras vacías/puntuación de cards_words
cards_filtered_words = FILTER cards_words BY NOT (word MATCHES '^[\\s.,!?;:\[\](){}\\"\’“”-]+$') AND NOT (word IN stopwords);

-- Cargar el dataset de transacciones limpio
transactions_cleaned_raw = LOAD '/output/transactions_cleaned/part-m-00000'
USING PigStorage(',') AS (
    id:int, date:chararray, client_id:int, card_id:int,
    amount:float, use_chip:chararray, merchant_id:int, merchant_city:chararray,
    merchant_state:chararray, zip:chararray, mcc:int, errors:chararray
);

-- Extraer campos textuales de transactions_cleaned y combinarlos
transactions_text = FOREACH transactions_cleaned_raw GENERATE
    CONCAT(CONCAT(CONCAT(use_chip, ' '), CONCAT(merchant_city, ' ')), CONCAT(merchant_state, ' '), errors) AS full_text;

-- Tokenizar las palabras de transactions_text
transactions_words = FOREACH transactions_text GENERATE FLATTEN(TOKENIZE(LOWER(full_text))) AS word;

-- Eliminar stopwords y palabras vacías/puntuación de transactions_words
transactions_filtered_words = FILTER transactions_words BY NOT (word MATCHES '^[\\s.,!?;:\[\](){}\\"\’“”-]+$') AND NOT (word IN stopwords);

-- Unir todas las palabras de ambos datasets
all_words = UNION cards_filtered_words, transactions_filtered_words;

-- Agrupar por palabra y contar ocurrencias
word_counts = GROUP all_words BY word;
counted_words = FOREACH word_counts GENERATE group AS word, COUNT(all_words) AS count;

-- Ordenar por conteo en orden descendente y obtener el top 3
sorted_words = ORDER counted_words BY count DESC;
top_3_words = LIMIT sorted_words 3;

-- Guardar el resultado en HDFS
STORE top_3_words INTO '/output/top_3_words' USING PigStorage(',');

Overwriting top_words_analysis.pig
